# GITS Scanner — PoC Report

**Purpose:** Revenue-weighted Google Trends index for nowcasting the chosen company's quarterly results.

**Reframe:** This is a NOWCASTING tool, not a leading-indicator tool. The target (quarterly earnings) is published ~30-45 days after the quarter end, so a calendar-coincident GITS at quarter-end gives meaningful lead time over the official 10-Q.

**Ticker selection:** the notebook reads `GITS_TICKER` env var (set by `gits report TICKER`).

**Prerequisites:**
1. `gits company add TICKER` registered
2. `gits segment add TICKER` for each business unit (≤ 5)
3. `reference/revenue_weights.csv` filled with at least 8 quarters
4. `gits fetch trends TICKER` and `gits fetch prices TICKER` executed

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

from gits.reference import list_tickers
from gits.storage.duckdb_io import get_conn, init_schema, read_trends, read_prices
from gits.engine.normalize import pivot_trends_wide, align_to_apple_fiscal_quarters
from gits.engine.weighting import load_weights_long, load_total_revenue, compute_gits_index
from gits.analysis.backtest import lead_lag_correlation, yoy_change, qoq_change, deseasonalize_fiscal
from gits.analysis.plots import three_axis_chart, segment_contribution_chart, lead_lag_chart

# Ticker selection: env var GITS_TICKER overrides the first registered ticker
TICKER = (os.environ.get('GITS_TICKER') or (list_tickers()[0] if list_tickers() else 'AAPL')).upper()
print(f'Running PoC for: {TICKER}')

pd.set_option('display.max_rows', 30)

## 1. Load all data

In [ ]:
conn = get_conn()
init_schema(conn)

trends_long = read_trends(conn, ticker=TICKER, geo='WW')
prices = read_prices(conn, ticker=TICKER).set_index('date')
weights_long = load_weights_long(TICKER)
csv_revenue = load_total_revenue(TICKER)

if trends_long.empty:
    raise RuntimeError(f'No trends data for {TICKER}. Run: gits fetch trends {TICKER}')
if weights_long.empty:
    raise RuntimeError(f'No revenue weights for {TICKER}. Edit reference/revenue_weights.csv')

print(f'Trends:        {len(trends_long)} rows, segments={trends_long["segment"].unique().tolist()}')
print(f'Prices:        {len(prices)} days, {prices.index.min()} → {prices.index.max()}')
print(f'CSV revenue:   {len(csv_revenue)} quarters, {csv_revenue.index.min().date()} → {csv_revenue.index.max().date()}')
print(f'Segment weights: {weights_long["quarter_end"].nunique()} fiscal quarters loaded')
csv_revenue.to_frame()

## 2. Compute GITS Index

In [ ]:
traffic_wide = pivot_trends_wide(trends_long)
gits = compute_gits_index(traffic_wide, weights_long)
gits.tail(10)

In [ ]:
fig = segment_contribution_chart(gits, title=f'{TICKER} — Weighted Segment Contribution to GITS')
fig.show()

## 3. Three-axis overlay: GITS vs Revenue vs Stock

In [ ]:
gits_q = gits['gits'].resample('QS').mean()
fig = three_axis_chart(
    gits=gits_q,
    revenue=csv_revenue,
    price=prices['adj_close'].resample('W').last(),
    title=f'{TICKER} — GITS Index vs Quarterly Revenue vs Stock Price',
)
fig.show()

## 4. Lead-lag correlation — THE PoC verdict

A positive `lead` value means GITS leads the target by N quarters.

In [ ]:
# Align GITS to the company's fiscal-quarter-end dates
fiscal_ends = list(csv_revenue.index)
traffic_at_fq = align_to_apple_fiscal_quarters(traffic_wide, fiscal_ends)
gits_fq = compute_gits_index(traffic_at_fq, weights_long)['gits'].dropna()

print(f'GITS at fiscal quarter-ends ({len(gits_fq)} quarters):')
display(gits_fq.to_frame('gits').join(csv_revenue.rename('revenue_usd_m')))

import numpy as np
from scipy.stats import pearsonr

def fq_lead_lag(leading: pd.Series, target: pd.Series, max_lead: int = 2) -> pd.DataFrame:
    """Pearson r at lead k between leading[t] and target[t+k]; both share a fiscal-quarter index."""
    aligned = pd.concat([leading.rename('leading'), target.rename('target')], axis=1, join='inner').sort_index()
    rows = []
    for k in range(-max_lead, max_lead + 1):
        shifted = aligned['target'].shift(-k)
        valid = pd.concat([aligned['leading'], shifted], axis=1).dropna()
        if len(valid) < 3 or valid.iloc[:, 0].std() == 0 or valid.iloc[:, 1].std() == 0:
            rows.append({'lead': k, 'n': len(valid), 'pearson_r': np.nan, 'p_value': np.nan})
            continue
        r, p = pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])
        rows.append({'lead': k, 'n': len(valid), 'pearson_r': r, 'p_value': p})
    return pd.DataFrame(rows)

rev_qoq = csv_revenue.pct_change()
gits_qoq = gits_fq.pct_change()

corr_rev = fq_lead_lag(gits_qoq, rev_qoq, max_lead=2)
print(f'\nGITS QoQ vs {TICKER} Revenue QoQ:')
print(corr_rev)
print('\n⚠️  Strong seasonality may dominate raw QoQ correlations; see deseasoned analysis below.')
lead_lag_chart(corr_rev, title=f'GITS QoQ → {TICKER} Revenue QoQ').show()

In [ ]:
price_at_fq = pd.Series({d: prices['adj_close'].asof(d) for d in fiscal_ends}).sort_index()
price_qoq = price_at_fq.pct_change()

corr_px = fq_lead_lag(gits_qoq, price_qoq, max_lead=2)
print(f'GITS QoQ vs {TICKER} Price QoQ:')
print(corr_px)
lead_lag_chart(corr_px, title=f'GITS QoQ → {TICKER} Price QoQ').show()

## 4b. Deseasonalized lead-lag — try to isolate the non-seasonal signal

The QoQ analysis above shows huge `lead=0` correlations (r=0.94 for price) that are mostly explained by Apple's strong seasonal cycle (Q1 holiday peak, Q3 trough). To find any TRUE leading information, we need to strip out the seasonal pattern.

**Method**: multiplicative deseasonalization — divide each observation by its fiscal-quarter-position factor (computed from this same data). With only 2 observations per quarter position, the seasonal estimate is noisy; treat this as a *directional* check, not a statistical test.

In [ ]:
gits_des = deseasonalize_fiscal(gits_fq, method='multiplicative')
rev_des = deseasonalize_fiscal(csv_revenue, method='multiplicative')
price_des = deseasonalize_fiscal(price_at_fq, method='multiplicative')

deseasoned_table = pd.concat([
    gits_fq.rename('GITS raw'),
    gits_des.rename('GITS deseasoned'),
    csv_revenue.rename('Revenue raw'),
    rev_des.rename('Revenue deseasoned'),
], axis=1).round(2)
print('Raw vs deseasoned levels (each pair should have flatter variance):')
display(deseasoned_table)

corr_rev_des = fq_lead_lag(gits_des, rev_des, max_lead=2)
corr_px_des = fq_lead_lag(gits_des, price_des, max_lead=2)

print(f'\nDeseasoned GITS LEVEL vs Deseasoned {TICKER} Revenue LEVEL:')
print(corr_rev_des)
print(f'\nDeseasoned GITS LEVEL vs Deseasoned {TICKER} Price LEVEL:')
print(corr_px_des)

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=gits_des.index, y=gits_des.values, name='GITS deseasoned', line=dict(color='#1f77b4', width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=rev_des.index, y=rev_des.values, name='Revenue deseasoned', line=dict(color='#2ca02c', width=2, dash='dot'), mode='lines+markers'), secondary_y=True)
fig.add_trace(go.Scatter(x=price_des.index, y=price_des.values, name='Price deseasoned', line=dict(color='#ff7f0e', width=1, dash='dash'), mode='lines+markers'), secondary_y=True)
fig.update_layout(title=f'{TICKER} — Deseasoned GITS vs Deseasoned Revenue vs Deseasoned Price',
                  yaxis=dict(title='GITS (deseasoned)', color='#1f77b4'),
                  yaxis2=dict(title='Revenue / Price (deseasoned)', overlaying='y', side='right'),
                  hovermode='x unified', height=480,
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

print('\nLead-lag charts:')
lead_lag_chart(corr_rev_des, title=f'Deseasoned GITS LEVEL → {TICKER} Revenue').show()
lead_lag_chart(corr_px_des, title=f'Deseasoned GITS LEVEL → {TICKER} Price').show()

## 5. Nowcasting verdict

**Interpretation guide (publication-lag adjusted)**:
- The target (quarterly revenue) is published ~30-45 days AFTER the quarter end
- A `lead = 0` correlation with quarter-end GITS is a **nowcasting** signal worth that publication lag in informational edge
- `lead = +1, +2` correlations indicate true forward prediction power (rare and noisy in small samples)
- `lead = -1, -2` correlations suggest GITS is responding to outcomes (reverse causality) — interpret with care

**Use cases for this report**:
1. **Nowcast revenue/EPS** before the official earnings release (5+ weeks before is common)
2. **Compare to analyst consensus** — the alpha is in beating the average estimate, not just predicting the raw number
3. **Drill down to individual segments** to identify which business unit is driving the move

**Data quality reminders**:
- Cross-segment RSV scale is best when all segments fit one pytrends query (≤5 segments)
- Statistical power is weak with < 16 quarters of data
- Seasonal estimates need ≥2 full annual cycles before they stabilize